## D3 - The Seeking Behavior
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 22, 2026

### About: Your robot can already avoid walls and red blocks. Today it does something different with blue blocks: instead of just turning away, it drives toward one, gets close enough to "touch" it, backs up, then turns.  
### This is the most complex single behavior of the week so far.

### One More Piece: What About "Touching" Something?
### Avoiding an obstacle just needs to know *something is there*. Seeking one out is harder: the robot needs to know when it's actually gotten close enough. How would you sense that?

### Why Not Motor Stall Detection?
### The RVR SDK can detect when a motor is physically stalled -- a genuine sensor reading, and the first approach we tried for sensing contact with a block.
### In testing, it turned out the competition blocks are too light to reliably trigger it: the robot can push a block without ever stalling. So that whole approach got dropped, not just deprioritized.
### 🤖 Real Robotics Engineering: When Your First Sensor Doesn't Work
### This is a completely normal part of real engineering -- teams routinely discover a planned sensing approach doesn't hold up once it meets real hardware. Figuring that out *is* part of the work, and falling back to a simpler, already-proven technique instead of chasing a fix is itself a valid engineering decision, not a consolation prize.

### The Fallback: Dead Reckoning to Estimate Touch
### Instead of sensing contact directly, today's robot drives forward an *estimated* distance -- using the same tools you already have from Tuesday's calibration in A3. It's an estimate, not a guarantee of contact -- the same "hope it's close enough" tradeoff dead reckoning always carries.
### After approaching, it backs up slightly (for a clean, predictable position) before turning.

### Full Behavior Spec
### - Avoid **wall** and **red** blocks -- exactly the same responses as D2, no changes there.
### - **Blue** blocks: instead of just turning away, drive toward it, approach a fixed distance, back up, then turn right.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY D3.1: Building the Seeking Behavior</span>

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 1. Import required libraries

import time
from sphero_sdk import RawMotorModesEnum
import ipywidgets as widgets
from IPython.display import display
from ipyfilechooser import FileChooser

from jetcam_lite import TraitletCamera, bgr8_to_jpeg

from robot_utils import get_rvr, close_if_exists
from jupyter_utils import register_dlink
from inference_utils import load_model_and_metadata, show_inference_grid
from behavior_utils import start_two_stage_behavior_loop, create_start_stop_buttons, stop_behavior_loop

### STEP 2. Choose both models -- same Free/Blocked and Wall/Red/Blue models you used in D2.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

print("Choose your FREE/BLOCKED model:")
free_blocked_chooser = FileChooser('/home/explorer/Models/')
free_blocked_chooser.filter_pattern = '*.pth'
display(free_blocked_chooser)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

print("Choose your WALL/RED/BLUE model:")
wall_red_blue_chooser = FileChooser('/home/explorer/Models/')
wall_red_blue_chooser.filter_pattern = '*.pth'
display(wall_red_blue_chooser)

### STEP 3. Load both models.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

free_blocked_model, free_blocked_device, free_blocked_classes, free_blocked_record = \
    load_model_and_metadata(free_blocked_chooser.selected)

print()

wall_red_blue_model, wall_red_blue_device, wall_red_blue_classes, wall_red_blue_record = \
    load_model_and_metadata(wall_red_blue_chooser.selected)

### Quick accuracy check on your Wall/Red/Blue model.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

show_inference_grid(
    wall_red_blue_model,
    wall_red_blue_classes,
    wall_red_blue_device,
    wall_red_blue_record['dataset_dir']
)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 4. Connect to the robot and start the camera.

rvr = get_rvr()
rvr.reset_yaw()

camera = TraitletCamera()
camera.start()

image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
display(image_widget)

### STEP 5. Enter your calibration from A3 -- check your spiral notebook for the `m` and `b` you calculated on Tuesday, and the speed/duty cycle you used during that calibration drive.
### These describe a straight-line relationship between distance and drive time:
```
time (seconds) = m * distance (meters) + b
```
### Note the units: your A3 calibration (and `time_for_distance()` below) works in **meters**, not centimeters -- `time_for_distance(0.5)` means 0.5 meters (50 cm). `seeking_behavior()`'s distances are given in cm since that's a more natural size to think about for this activity, so you'll need to convert (divide by 100) before calling `time_for_distance()`.
### This only works if `seeking_speed` matches the speed your `m`/`b` were actually calibrated at -- a different speed makes the equation meaningless.

In [ ]:
##### ----- FEEL FREE TO CHANGE THESE VALUES ----- #####
# Your calibration from A3 -- get these from your spiral notebook!
# Named m and b (not calibration_m/calibration_b) so they match what
# time_for_distance() expects -- same names you used in A3.
m = 0.55   # replace with your m from A3
b = 0.14   # replace with your b from A3
seeking_speed = 100   # must match the speed/duty cycle you used during A3 calibration

### STEP 6. Write `seeking_behavior()`. `time_for_distance()` is already provided above it -- the same function from A3, using your `m`/`b` to convert a distance in **meters** into a drive time in seconds. It's called once each time the robot needs to approach and "touch" a blue block, and should:
1. Call `time_for_distance(forward_distance_cm / 100)` (dividing by 100 converts cm to meters) to get a drive time, and drive forward that long at `seeking_speed`.
2. Do the same for `backup_distance_cm`, and drive **backward** that long -- same heading, but with the reverse flag (`1` instead of `0`) so the robot backs up without spinning around.
3. Update `heading_state['current']` to `90`, then turn using `drive_with_heading()` -- same pattern as D2's red/blue turns.

### Note: this function uses `time.sleep()` between each step, so it *blocks* while it runs -- the whole approach-back up-turn sequence happens as one call, rather than spreading across multiple loop cycles. That's intentional. One thing to watch for: if your calibration implies a very slow speed, a single leg's sleep time could approach the RVR's own 2-second command timeout. If your robot ever stops unexpectedly mid-seek, that's likely why -- try a higher `seeking_speed`.

In [ ]:
#### ------> ACTIVITY D3.1: seeking_behavior function <-----#####
# About: called from decide_action's 'blue' branch. Approaches a blue
# block using dead reckoning (m, b, seeking_speed from above), since
# motor-stall detection isn't reliable on these blocks.

##### INSTRUCTIONS: #####
# 1. Call time_for_distance() to convert forward_distance_cm to a drive
#    time -- remember it expects METERS, so divide by 100 first.
#    Drive forward that long at seeking_speed.
# 2. Do the same for backup_distance_cm, and drive backward that long
#    (same heading, reverse flag = 1).
# 3. Update heading_state['current'] to 90, then turn.

# This is the same function used in the calibration notebook, it uses your m and b values to compute the time required to achieve a given distance
# Remember how to call it? Example: time_to_drive = time_for_distance(0.5)    # This gives us the duration to drive 0.5 meters or 50 cm
def time_for_distance(goal_distance):
    # Apply the line-of-best-fit equation: time = m * distance + b
    drive_time = m * goal_distance + b
    return drive_time


def seeking_behavior(rvr):
    forward_distance_cm = 20
    backup_distance_cm = 10

    #<<<<<< replace this with your code >>>>>>
    pass

### STEP 7. `decide_action` below reuses D2's wall and red responses exactly as you built them -- no changes needed there. Fill in just the one line in the `'blue'` branch: call the function you just wrote.

In [ ]:
#### ------> ACTIVITY D3.2: decide_action function <-----#####
# About: primary_label is the Free/Blocked prediction (always present).
# secondary_label is the Wall/Red/Blue prediction -- only present
# (not None) when primary_label == 'blocked'.

heading_state = {'current': 0}

def decide_action(rvr, primary_label, secondary_label):
    if primary_label == 'free':
        rvr.drive_with_heading(100, heading_state['current'], 0)

    else:  # primary_label == 'blocked'
        if secondary_label == 'wall':
            heading_state['current'] = (heading_state['current'] + 180) % 360
            rvr.drive_with_heading(100, heading_state['current'], 0)

        elif secondary_label == 'red':
            heading_state['current'] = 270
            rvr.drive_with_heading(100, heading_state['current'], 0)

        else:  # secondary_label == 'blue'
            #<<<<<< replace this with your code >>>>>>
            pass

### STEP 8. Build the Start/Stop buttons. Running this cell does **not** move the robot -- nothing happens until you press Start.

######## WILL CAUSE ROBOT MOTION ONCE STARTED: ENSURE ROBOT IS ON THE GROUND, WITH SPACE TO APPROACH A BLUE BLOCK, AND WALL/RED OBSTACLES NEARBY TO TEST AGAINST #########

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

buttons = create_start_stop_buttons(
    lambda: start_two_stage_behavior_loop(
        rvr, camera,
        free_blocked_model, free_blocked_classes, free_blocked_device,
        wall_red_blue_model, wall_red_blue_classes, wall_red_blue_device,
        decide_action,
        trigger_label='blocked'
    )
)
display(buttons)

### Press **Start**, then test all three: a wall, a red block, and a blue block. For blue, watch whether the approach distance looks about right -- close enough to "touch," not a collision and not stopping short. Press **Stop** any time to immediately halt the behavior.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 9. Stop the behavior.
stop_behavior_loop()

<span style="color: green; font-size: 55px; font-style: italic;">Student Discussion Time</span>

### Talk through these questions with your team:
### - How close did the approach get? Did you need to adjust `seeking_speed`, or your remembered `m`/`b`, to get a good result?
### - This is the second time this week you've used dead reckoning to estimate distance -- Tuesday's navigation course, and today's approach distance. What's different about how much it matters if today's estimate is a little off, compared to Tuesday's?
### - Your A3 calibration is three days old. Do you think it's still accurate? What could have changed since then (battery wear, wheel grip, the surface you're driving on)?
### - What would you need to add to make this more reliable than a fixed-distance guess?

## Outstanding work -- your robot now avoids obstacles, classifies what it sees, and seeks out a target, all in one integrated behavior.

## **NEXT**: Tomorrow is the final challenge -- searching an arena, telling single blocks from stacks, and combining everything you've built this week into one autonomous program.

# SOLUTIONS:

### ------> SOLUTION - Activity D3.1: seeking_behavior function <-----

In [ ]:
#### ------> SOLUTION - Activity D3.1: seeking_behavior function <-----#####

def time_for_distance(goal_distance):
    # Apply the line-of-best-fit equation: time = m * distance + b
    drive_time = m * goal_distance + b
    return drive_time


def seeking_behavior(rvr):
    forward_distance_cm = 20
    backup_distance_cm = 10

    # GEORGE's Solution:
    # time_for_distance() expects meters -- divide our cm distances by
    # 100 before calling it.
    forward_time = time_for_distance(forward_distance_cm / 100)
    rvr.drive_with_heading(seeking_speed, heading_state['current'], 0)
    time.sleep(forward_time)

    backup_time = time_for_distance(backup_distance_cm / 100)
    rvr.drive_with_heading(seeking_speed, heading_state['current'], 1)  # reverse flag
    time.sleep(backup_time)

    heading_state['current'] = 90
    rvr.drive_with_heading(seeking_speed, heading_state['current'], 0)

### ------> SOLUTION - Activity D3.2: decide_action function <-----

In [ ]:
#### ------> SOLUTION - Activity D3.2: decide_action function <-----#####

heading_state = {'current': 0}

def decide_action(rvr, primary_label, secondary_label):
    if primary_label == 'free':
        rvr.drive_with_heading(100, heading_state['current'], 0)

    else:  # primary_label == 'blocked'
        if secondary_label == 'wall':
            heading_state['current'] = (heading_state['current'] + 180) % 360
            rvr.drive_with_heading(100, heading_state['current'], 0)

        elif secondary_label == 'red':
            heading_state['current'] = 270
            rvr.drive_with_heading(100, heading_state['current'], 0)

        else:  # secondary_label == 'blue'
            # GEORGE's Solution:
            seeking_behavior(rvr)

### Wrapping Up
Before you move on to the next notebook, run the cell below to release the robot's connection.

In [ ]:
#### ------> RUN THIS CELL WHEN YOU'RE DONE WITH THIS NOTEBOOK <-----#####
close_if_exists()
print("Robot connection closed. Safe to move on to the next notebook!")